# LC 33 — Search in Rotated Sorted Array
**Day-78 | Binary Search on Rotated Arrays | Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">

**Core Insight:** A rotated sorted array is always made of **two**
sorted halves. At every mid-point, at least one half is guaranteed
to be sorted. Identify which half is sorted, check if target lives
there, and discard the other half — giving O(log n) search.

</div>

## Official Problem Statement

Given an integer array `nums` sorted in ascending order and rotated
at some unknown pivot index, and an integer `target`, return the
index of `target` if it is in `nums`, or `-1` if it is not.

You must write an algorithm with **O(log n)** runtime complexity.

**Constraints:**
- `1 <= nums.length <= 5000`
- `-10^4 <= nums[i], target <= 10^4`
- All values of `nums` are **unique**.
- `nums` is an ascending array rotated at some pivot.

## What This Is Actually Asking

Imagine a sorted list `[1,2,3,4,5,6,7]` that someone cut at index 3
and moved the front to the back: `[4,5,6,7,1,2,3]`.  
Now find a target value — without scanning every element.

A naive scan is O(n). The trick is that **even after rotation**,
binary search still works — you just need one extra check per step:
figure out which of the two halves around `mid` is the sorted one,
then decide which side the target must be on.

## Walk Through an Example by Hand

```
nums   = [4, 5, 6, 7, 0, 1, 2]   target = 0
indices:  0  1  2  3  4  5  6
```

**Step 1:** lo=0, hi=6, mid=3  
- nums[mid]=7, nums[lo]=4 → 4<=7 → **left half sorted**  
- Is 0 in [4..7)? No → go right: lo=4

**Step 2:** lo=4, hi=6, mid=5  
- nums[mid]=1, nums[lo]=0 → 0<=1 → **left half sorted**  
- Is 0 in [0..1)? Yes → go left: hi=4

**Step 3:** lo=4, hi=4, mid=4  
- nums[mid]=0 == target → **return 4** ✓

---

```
nums   = [4, 5, 6, 7, 0, 1, 2]   target = 3
```

**Step 1:** lo=0, hi=6, mid=3 → left sorted, 3 not in [4..7) → lo=4  
**Step 2:** lo=4, hi=6, mid=5 → left sorted, 3 not in [0..1) → hi=4  
**Step 3:** lo=4, hi=4, mid=4 → nums[4]=0 ≠ 3 → right sorted  
  - 3 in (0..0]? No → hi=3  
  - lo>hi → **return -1** ✓

## The Picture

```
Array (rotated at index 4):

 idx:  0    1    2    3    4    5    6
      [4,   5,   6,   7,   0,   1,   2]
       ^              ^              ^
       lo            mid            hi

  nums[lo]=4 <= nums[mid]=7  →  LEFT side is sorted

  ┌─ sorted ─────────┐   ┌─ contains pivot ─┐
  [ 4    5    6    7  |   0    1    2 ]
    lo            mid     mid+1      hi

  If target ∈ [nums[lo], nums[mid])  →  hi = mid - 1
  Else                               →  lo = mid + 1

After lo=4:

 idx:  4    5    6
      [0,   1,   2]
       ^    ^    ^
       lo  mid   hi

  nums[lo]=0 <= nums[mid]=1  →  LEFT side is sorted
  target=0 ∈ [0, 1)          →  hi = mid - 1 = 4
  lo==hi==4, nums[4]=0 → FOUND
```

## When To Use This Pattern

Use **Binary Search on Rotated Array** when:

- The array is sorted but may be rotated at an unknown pivot.
- All values are **unique** (no duplicates) — LC 33.
- You need O(log n) time (linear scan is too slow).

**Related patterns:**
- Find Minimum in Rotated Sorted Array (LC 153) — same structure.
- Search in Rotated Sorted Array II (LC 81) — adds duplicates.
- Binary search on answer space (different flavor).

**Do NOT use when:**
- The array has duplicates (use LC 81 logic instead).
- The array is not sorted at all.

## The Approach

```
Standard binary search skeleton, one extra decision per step:

while lo <= hi:
    mid = (lo + hi) // 2
    if nums[mid] == target: return mid

    # Which half is sorted?
    if nums[lo] <= nums[mid]:          # left half sorted
        if nums[lo] <= target < nums[mid]:
            hi = mid - 1               # target in left
        else:
            lo = mid + 1               # target in right
    else:                              # right half sorted
        if nums[mid] < target <= nums[hi]:
            lo = mid + 1               # target in right
        else:
            hi = mid - 1               # target in left

return -1
```

**Key invariant:** `nums[lo] <= nums[mid]` tells us the left half
has no rotation break, so it is a clean sorted segment.

In [1]:
from typing import List

In [2]:
def test_harness(func):
    """Run test cases and print PASSED / FAILED summary."""
    cases = [
        # (nums, target, expected)
        ([4, 5, 6, 7, 0, 1, 2], 0,  4),
        ([4, 5, 6, 7, 0, 1, 2], 3, -1),
        ([1],                   0, -1),
        ([1],                   1,  0),
        ([3, 1],                1,  1),
        ([3, 1],                3,  0),
        ([5, 1, 3],             3,  2),
        ([1, 3, 5],             5,  2),   # no rotation
        ([6, 7, 1, 2, 3, 4, 5], 7,  1),
    ]
    passed = 0
    for nums, target, expected in cases:
        result = func(nums, target)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(
                f"  {status}: nums={nums} target={target} "
                f"expected={expected} got={result}"
            )
    total = len(cases)
    print(f"\nResult: {passed}/{total} tests passed.")

In [5]:
def search(nums: List[int], target: int) -> int:
    """
    LC 33 — Search in Rotated Sorted Array.

    A sorted array is rotated at an unknown pivot. Find target.
    All values are unique.

    Strategy:
    ---------
    At every mid, one of the two halves is guaranteed sorted.
    Determine which half is sorted (nums[lo] <= nums[mid] means
    left is sorted). Check if target falls in that sorted half.
    If yes, discard the other half. If no, discard this half.

    Args:
        nums:   Rotated sorted array with unique values.
        target: Integer value to find.

    Returns:
        Index of target in nums, or -1 if not found.

    Examples:
        >>> search([4,5,6,7,0,1,2], 0)
        4
        >>> search([4,5,6,7,0,1,2], 3)
        -1
    """
    l , r = 0, len(nums) -1

    while l <= r:
        mid = l + (r-l)//2
        if nums[mid] == target:
            return mid

        if nums[l] <= nums[mid]:
            if target > nums[mid] or target < nums[l]:
                l = mid + 1
            else:
                r = mid - 1
        else:
            if target < nums[mid] or target > nums[r]:
                r = mid -1
            else:
                l = mid + 1
    return -1

test_harness(search)
print(search([4,5,6,7,0,1,2], 0))   # 4
print(search([4,5,6,7,0,1,2], 3))   # -1
print(search([1], 0))                # -1
print(search([3,1], 1))             # 1


Result: 9/9 tests passed.
4
-1
-1
1


In [ ]:
# Uncomment and run when solution is ready
# test_harness(search)

## Complexity

| Dimension | Value | Reason |
|-----------|-------|--------|
| Time  | O(log n) | Search space halves every iteration |
| Space | O(1)     | Only pointer variables used          |

**Why not O(n)?**  
Each iteration we conclusively eliminate one half of the remaining
array, exactly like standard binary search — the rotation only adds
a constant-time check to pick the correct half.

## Real World Connection

**Version rollback in a deployment pipeline:**  
Imagine a ring buffer of build versions ordered by timestamp, but
the pointer has wrapped around midnight — the newest build is at
index 0, and indices 1..N are older builds from yesterday. You want
to locate build ID 9312 quickly without scanning all N entries.
The same two-half analysis applies: compare the start and midpoint
timestamps to determine which half is continuous, then binary-search
within the correct segment — O(log N) instead of O(N).

> **Simplicity and clarity is Gold.** — Sean's Study Mantra